# Comparar Bond, Rittinger y Kick en una reducción

**Pregunta:** ¿Dónde coinciden Bond, Rittinger y Kick y cómo divergen sus predicciones fuera de la calibración?

Nivel: 3. Requisitos previos: Niveles 1 y 2 de este tema. Tiempo de lectura: 24 minutos.



## Objetivos

Comparar tres hipótesis de conminución en el mismo rango de alimentación y producto, identifica el punto de calibración donde coinciden y establece qué permite y qué no permite concluir la comparación.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Para la lección de nivel 3 sobre 01-comminution, registra las versiones fijadas antes de ejecutar el análisis; así se puede reconstruir el entorno que produjo cada tabla y figura.

In [ ]:
# Importar herramientas de análisis y registrar sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Datos

Responsable: autores de OreFlow, licencia MIT. El catálogo aporta mineral, triturador, molino, clasificador y operación nominal del pórfido blando. Las constantes y extremos de referencia provienen de engine/data/constants.json. Salidas y distribuciones de tamaño se calculan aquí, sin cargar una traza guardada. No se introduce distribución medida, campaña energética ni otro conjunto externo. Fuente: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Método

Bond: $W=W_i(10/\sqrt{P}-10/\sqrt{F})$. Rittinger: $E=K_R(1/P-1/F)$; Kick: $E=K_K\ln(F/P)$. OreFlow calibra $K_R$ y $K_K$ para que las tres leyes coincidan en la reducción de referencia declarada: 10.000 µm a 150 µm. Fuera de ese punto representan hipótesis de reducción de tamaño distintas. Aquí `law_curves` es la implementación autoritativa.

## Comparación

El índice de trabajo Bond es un método empírico de circuito estándar para reducción por tamaño, formalizado en la guía Bond Efficiency de Global Mining Guidelines Group (GMG01-MP-2021). Rittinger y Kick representan las hipótesis de superficie y razón de reducción; ninguna es válida universalmente en todos los regímenes de tamaño. OreFlow reporta Bond y mantiene las otras leyes como comparaciones diagnósticas. Es una comparación de ecuaciones en un caso de autoría, no una afirmación de predicción superior.

La comparación física usa F80 de alimentación al molino recién resuelto. El F80 del triturador corresponde a una reducción anterior y no se sustituye en este cálculo. La reducción de referencia separada determina constantes diagnósticas, sin redefinir el circuito operativo.

In [ ]:
# La comparación física usa F80 de alimentación al molino recién resuelto
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.engine.energy import law_curves, reference_constants
from pipeline.engine.constants import constant

case = CASE_BY_ID["copper_porphyry_soft"]
nominal = simulate(case.ore, case.plant, case.nominal)
metrics = nominal.metrics
wi = case.nominal.work_index_kwh_t
f80 = nominal.grinding.feed_f80_um
print("F80 alcanzado del molino (µm):", round(f80, 3))
print("P80 alcanzado (µm):", round(nominal.grinding.p80_um, 3))


Las curvas operativas comparten alimentación alcanzada e índice de trabajo. Otro cálculo evalúa las tres leyes exactamente en alimentación y producto de referencia e incorpora ese producto exacto a su propia malla. La igualdad allí procede de calibración; no se impone igualdad a 150 µm con otra alimentación.

In [ ]:
# Las curvas operativas comparten alimentación alcanzada e índice de trabajo
p80 = np.geomspace(50.0, f80 / 1.01, 80)
curves = law_curves(wi, f80, p80)
reference_feed = float(constant("bond.reference_feed_um"))
reference_product = float(constant("bond.reference_product_um"))
calibration = law_curves(wi, reference_feed, np.array([reference_product]))
reference_products = np.unique(
    np.r_[np.geomspace(50.0, reference_feed / 1.01, 80), reference_product]
)
reference_curves = law_curves(wi, reference_feed, reference_products)
display(pd.DataFrame(calibration).round(6))


El panel izquierdo evalúa la reducción física del molino. El derecho es un experimento de calibración distinto y rotulado; cada marca usa el valor exacto de su ley en el extremo de referencia. Ninguna marca toma la ordenada de un tamaño vecino.

In [ ]:
# El panel izquierdo evalúa la reducción física del molino
plt.rcParams.update({"font.size": 22, "axes.titlepad": 40})
fig, axes = plt.subplots(2, 1, figsize=(6.4, 4.8))
for name in ("bond", "rittinger", "kick"):
    axes[0].plot(curves["p80_um"], curves[name], label=name.title())
    axes[1].plot(reference_products, reference_curves[name], label=name.title())
    axes[1].scatter([reference_product], [calibration[name][0]], s=28)
axes[0].set(
    title=f"Alimentación física al molino: F80={f80:.1f} µm", ylabel="Energía\nespecífica\n(kWh/t)"
)
axes[1].set(
    title=f"Calibración: F80={reference_feed:g} µm; P80={reference_product:g} µm",
    ylabel="Energía\nespecífica\n(kWh/t)",
)
for axis in axes:
    axis.set_xscale("log")
    axis.set_xlabel("P80 de producto (µm)")
    axis.invert_xaxis()
    axis.grid(alpha=0.25)
    axis.legend()
fig.subplots_adjust(left=0.24, right=0.96, bottom=0.15, top=0.91, hspace=1.0)


Las curvas izquierdas pueden diferir a 150 µm porque usan alimentación alcanzada del molino. Las marcas derechas coinciden en la reducción de referencia declarada, evaluada exactamente y no desde un punto vecino. Son preguntas distintas con extremos identificados. La tabla nominal y el barrido relativo posteriores mantienen la base física del molino; calibrar no aporta otro estado operativo ni validación energética medida.

Cuantifica la dispersión en la reducción real del caso. Interprétala como desacuerdo estructural entre hipótesis, no como intervalo de confianza ni error observado en planta.

In [ ]:
# Cuantifica la dispersión en la reducción real del caso.
actual_p80 = float(metrics["p80_um"])
point = law_curves(wi, f80, np.array([actual_p80]))
comparison = pd.DataFrame(
    {
        "ley": ["Bond", "Rittinger", "Kick"],
        "energy_kwh_t": [point["bond"][0], point["rittinger"][0], point["kick"][0]],
    }
)
comparison["relativo_a_bond"] = comparison.energy_kwh_t / comparison.energy_kwh_t.iloc[0]
display(comparison)


Resolver el circuito nominal antes de evaluar hipótesis alternativas de energía. Los tamaños de alimentación y producto alcanzados, además del objetivo solicitado, definen una reducción física. Bond, Rittinger y Kick usan el mismo índice de trabajo y los mismos extremos en este experimento. Su calibración coincide en una reducción especificada, pero esa coincidencia es impuesta y no valida su respuesta fuera de ella.

In [ ]:
# Resolver el circuito nominal antes de evaluar hipótesis alternativas de energía.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.engine.energy import law_curves

case = CASE_BY_ID["copper_porphyry_soft"]
nominal = simulate(case.ore, case.plant, case.nominal)
f80 = nominal.grinding.feed_f80_um
wi = case.nominal.work_index_kwh_t
print("F80 alcanzado del molino (µm):", round(f80, 2))


Calcular el trabajo calibrado al tamaño de producto alcanzado. Son tres requerimientos alternativos, no componentes de energía que deban sumarse. Compararlos con energía neta de molienda exige bases coherentes de circuito y eficiencia. Conservar los valores en kWh por tonelada de alimentación fresca para que una comparación posterior de potencia no use por accidente el tonelaje recirculante del molino.

In [ ]:
# Calcular el trabajo calibrado al tamaño de producto alcanzado.
at_product = law_curves(wi, f80, np.array([nominal.grinding.p80_um]))
display(pd.DataFrame(at_product).round(3))


Solicitar al motor un producto más fino manteniendo fijos alimentación fresca y mineral. Así se aísla la demanda de liberación del tratamiento. Un molino limitado por potencia puede no alcanzar el objetivo solicitado; por eso la tabla conserva tamaños solicitado y alcanzado. El objetivo más fino es una perturbación controlada del modelo, no una recomendación operativa para este mineral.

In [ ]:
# Solicitar al motor un producto más fino manteniendo fijos alimentación fresca y mineral.
fine_point = case.nominal.with_values(target_p80_um=case.nominal.target_p80_um * 0.8)
fine = simulate(case.ore, case.plant, fine_point)
print("P80 solicitado (µm):", fine_point.target_p80_um)
print("P80 alcanzado (µm):", round(fine.grinding.p80_um, 3))
print("Potencia requerida (kW):", round(fine.grinding.required_power_kw, 3))


Repetir con un objetivo más grueso para rodear el punto nominal. Engrosar puede reducir la demanda de molienda y dejar más mineral valioso sin liberar. El circuito completo calcula esa consecuencia, en vez de asignar recuperación desde una curva de energía. Ordenar solo por energía omitiría la pérdida de metal recuperado; la comparación final requiere también recuperación y potencia.

In [ ]:
# Repetir con un objetivo más grueso para rodear el punto nominal.
coarse_point = case.nominal.with_values(target_p80_um=case.nominal.target_p80_um * 1.2)
coarse = simulate(case.ore, case.plant, coarse_point)
print("Recuperación con molienda gruesa (%):", round(coarse.metrics["recovery_pct"], 3))
print("Potencia con molienda gruesa (kW):", round(coarse.grinding.required_power_kw, 3))


Evaluar las tres leyes en el mismo barrido logarítmico de producto. La alimentación queda fija y cada producto solicitado es más pequeño. Normaliza cada alternativa por Bond después de calcular su energía: los cocientes ocultan la demanda absoluta. Un cruce representa acuerdo de ecuaciones calibradas en un tamaño, no una transición física ni evidencia de que sus supuestos sean intercambiables.

In [ ]:
# Evaluar las tres leyes en el mismo barrido logarítmico de producto.
products = np.geomspace(max(50.0, f80 / 30), f80 / 1.1, 40)
sweep = law_curves(wi, f80, products)
ratio = {
    name: np.asarray(sweep[name]) / np.asarray(sweep["bond"]) for name in ("rittinger", "kick")
}
print("Máximo Rittinger / Bond:", round(float(max(ratio["rittinger"])), 3))


Graficar la divergencia relativa de las leyes para distinguir coincidencia de calibración y forma de cada hipótesis de reducción.

In [ ]:
# Graficar la divergencia relativa de las leyes para distinguir coincidencia de calibración y forma
# de cada hipótesis de reducción..
plt.rcParams.update({"font.size": 18, "axes.titlepad": 64})
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for name, values in ratio.items():
    ax.plot(products, values, label=name.title())
ax.axhline(1.0, color="black", linestyle="--", label="Bond")
ax.set(
    xscale="log",
    xlabel="P80 de producto (µm)",
    ylabel="Energía / Bond\n(1)",
    title="Relaciones de leyes para alimentación\nal molino",
)
ax.legend(ncol=3, **legend_style)
fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.61)
ax.locator_params(axis="x", numticks=4)
ax.tick_params(axis="x", which="minor", labelbottom=False)


Los cocientes muestran dónde una hipótesis de superficie o de razón de reducción predice una demanda distinta de Bond. La unidad es referencia comparativa, no intervalo de confianza. El barrido mantiene fijos alimentación e índice de trabajo, por lo que no explica cambios geológicos de dureza. Usar los circuitos fino y grueso resueltos para evaluar si el producto deseado es alcanzable con la potencia instalada antes de tratar un requerimiento de energía como estado realizable.

Una igualdad de calibración exige dos extremos declarados, no solo índice de trabajo. OreFlow determina constantes Rittinger y Kick con la misma reducción de referencia Bond y las conserva en el barrido. Cambiar la referencia cambiaría el desacuerdo numérico sin cambiar hipótesis matemáticas. Al repetir con un ensayo mineral, registra convenciones de tamaños, base energética y reducción de calibración. Coincidir en un punto elegido para calibrar no aporta una observación independiente adicional.

## Resultados

**Respuesta:** Con el mismo par alimentación/producto nominal, Bond predice <!-- result:Trabajo Bond (kWh/t) -->0<!-- /result --> kWh/t, Rittinger <!-- result:Trabajo Rittinger (kWh/t) -->0<!-- /result --> y Kick <!-- result:Trabajo Kick (kWh/t) -->0<!-- /result -->. La igualdad de referencia se impone al calibrar; el barrido muestra divergencia fuera de ella. Se comparan ecuaciones en mineral de autoría, sin validación contra energía medida.

Vincular valores de leyes en extremos físicos del molino por separado de la tabla de calibración exacta anterior.

In [ ]:
# Resumir las salidas calculadas que sostienen la interpretación de esta lección.
answer_values = {
    "P80 nominal (µm)": nominal.grinding.p80_um,
    "P80 fino solicitado (µm)": fine_point.target_p80_um,
    "P80 fino alcanzado (µm)": fine.grinding.p80_um,
    "Potencia fina requerida (kW)": fine.grinding.required_power_kw,
    "Trabajo Bond (kWh/t)": at_product["bond"][0],
    "Trabajo Rittinger (kWh/t)": at_product["rittinger"][0],
    "Trabajo Kick (kWh/t)": at_product["kick"][0],
}
answer_values = {label: round(float(number), 3) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Resumen

La comparación controlada en el mismo caso hace visible la selección del modelo: calibrar hace coincidir las leyes en un punto, no en todos. Siguen aplicando la calificación de finura de Bond y su base energética.

## Referencias

- https://www.globalminingguidelines.org/guidelines/bond-efficiency/
- https://doi.org/10.1016/0301-7516(80)90034-4
- https://doi.org/10.1016/0301-7516(80)90034-4